<a href="https://colab.research.google.com/github/LuisFelipeVelasco/ML_And_Generative_AI_Experiments/blob/main/Notebooks/neural_network_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NEURAL NETWORK FROM SCRATCH**

**IMPORT LIBRARIES**

In [3]:
import numpy as np
import pandas as pd

**DEFINE CLASS**

In [249]:
class NeuralNetwork():

  """Representation of a neural network"""

  def __init__(self, learning_rate, layouts, epochs):

    """Initialize the neural network"""

    self.learning_rate = learning_rate
    self.layouts = layouts
    self.epochs = epochs

  def initialize_weights(self ,number_of_inputs_of_network):

    """Initialize the weights of the neural network with random numbers between 1 and 0"""

    self.weights = []
    for current_layer in range(len(self.layouts)):
      number_of_neurons = self.layouts[current_layer]
      if current_layer == 0:
        shape=(number_of_neurons, number_of_inputs_of_network)
        self.weights.append(np.random.random_sample(shape))
      else:
        layer_before = current_layer-1
        number_of_inputs_of_layer = self.layouts[layer_before]
        shape=(number_of_neurons ,number_of_inputs_of_layer)
        self.weights.append(np.random.random_sample(shape))

  def initialize_biases(self):

    """Initialize the biases of the neural network with random numbers between 1 and 0"""

    self.biases=[]
    for current_layer in range(len(self.layouts)):
      number_of_neurons = self.layouts[current_layer]
      self.biases.append(np.random.randn(number_of_neurons,1))

  def get_activation_of_net_input(self,z):

    """Get the activation of the net input of a neural network"""

    return 1/(1+np.exp(-z))

  def get_neurons_activations(self,sample_features):

    """Get the activation of the neurons of the neural network"""
    current_activations = sample_features
    neurons_activations = []
    for current_layer in range(len(self.layouts)):
      current_activations= self.get_activation_of_net_input((self.weights[current_layer] @ current_activations)+self.biases[current_layer])
      neurons_activations.append(current_activations)
    return neurons_activations

  def backpropagation(self,neurons_activations, sample_features, prediction_error):

    """Calculate the gradient of the weights and biases"""

    number_of_layers= len(self.layouts)
    weights_gradients=[]
    biases_gradients=[]
    for current_layer in range(number_of_layers-1,-1,-1):
      neurons_activations_current_layer= neurons_activations[current_layer]
      sensibility_of_activation_with_respect_to_net_input = neurons_activations_current_layer*(1-neurons_activations_current_layer)
      if(current_layer == number_of_layers-1):sensibility_of_cost_with_respect_to_activation =-1*prediction_error
      else: sensibility_of_cost_with_respect_to_activation = np.transpose(self.weights[current_layer+1]) @ neurons_errors
      if(current_layer==0): input_neurons= np.transpose(sample_features)
      else:input_neurons= np.transpose(neurons_activations[current_layer-1])
      neurons_errors=sensibility_of_cost_with_respect_to_activation * sensibility_of_activation_with_respect_to_net_input
      sensibility_of_cost_with_respect_to_weights = neurons_errors @ input_neurons
      sensibility_of_cost_with_respect_to_biases=neurons_errors
      weights_gradients.insert(0,sensibility_of_cost_with_respect_to_weights)
      biases_gradients.insert(0,sensibility_of_cost_with_respect_to_biases)

    return weights_gradients, biases_gradients


  def optimization(self,weights_gradients,biases_gradients):

    "update the weights and the biases of the neural network to minimize the cost using gradient descent"

    number_of_layers= len(self.layouts)
    for current_layer in range(number_of_layers):
      self.weights[current_layer] += -1*self.learning_rate*weights_gradients[current_layer]
      self.biases[current_layer] += -1*self.learning_rate*biases_gradients[current_layer]

  def fit(self, samples,labels):

    """Train neural network"""

    number_of_samples = samples.shape[0]
    number_of_features= samples.shape[1]
    self.cost = 0;
    self.initialize_weights(number_of_features)
    self.initialize_biases()
    for i in range (self.epochs):
      for j in range (number_of_samples):
        sample = samples[j][:,None]
        label= labels[j]
        neurons_activations=self.get_neurons_activations(sample)
        prediction_error=np.transpose(label)-neurons_activations[-1]
        self.cost+=prediction_error**2
        weights_gradients, biases_gradients= self.backpropagation(neurons_activations,sample,prediction_error)
        self.optimization(weights_gradients,biases_gradients)
      self.cost/=number_of_samples
      print(self.cost)





# **TEST NEURAL NETWORK**

**CHARGE SOCCER DATASET**

In [112]:
!wget https://raw.githubusercontent.com/LuisFelipeVelasco/ML_And_Generative_AI_Experiments/main/Sources/historical_european_soccer_matches.csv
df=pd.read_csv('historical_european_soccer_matches.csv')

--2026-08-08 19:03:39--  https://raw.githubusercontent.com/LuisFelipeVelasco/ML_And_Generative_AI_Experiments/main/Sources/historical_european_soccer_matches.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 105787 (103K) [text/plain]
Saving to: ‘historical_european_soccer_matches.csv.3’

historical_european 100%[===================>] 103.31K  --.-KB/s    in 0.02s   

2026-08-08 19:03:39 (6.05 MB/s) - ‘historical_european_soccer_matches.csv.3’ saved [105787/105787]



In [272]:
samples= df.iloc[:,:2].values
labels= df.iloc[:,-1].values
model= NeuralNetwork(0.1,[55,25,12,6,3,1],5)
model.fit(samples,labels)

[[0.25142082]]
[[0.25118924]]
[[0.25096376]]
[[0.25071002]]
[[0.25050688]]


In [276]:
sample=np.array([1,4])[:,None]
print(model.get_neurons_activations(sample)[-1])

[[0.46741648]]
